# HybridGuard — CANOPI Orchestrator (NPL)

Trains the CANOPI projection head (frozen L0 canonicalizer + frozen encoder
+ JOINT objective) over 5 seeds and runs E1, E3–E7, E10. Imports the installed
`hybridguard.canopi` package and REUSES `canopi.data` (orchestrator 60/20/20 @
seed 1337) + `canopi.eval` (frozen-τ protocol). No LLM at inference.

**Run on Colab Pro+ (GPU):** Runtime → GPU, then Run all. Outputs land in
`runs/<run_id>/` (per-seed CSV + aggregated mean±std + LaTeX tables).


## 0 · Config — edit once


In [ ]:
REPO_URL = 'https://github.com/ShaikhaTheGreen/HybridGuard.git'
BRANCH   = 'canopi-npl'
CONFIG   = 'configs/canopi_main.yaml'   # or a baselines/ ablations/ yaml
MOUNT_DRIVE = True                       # persist runs/ to Drive
DRIVE_DIR   = '/content/drive/MyDrive/HybridGuard'
RUN_ID   = None                          # None -> derived from config name + date
SEEDS    = [42, 2025, 7, 1337, 314]      # pre-registered; do not change
SMOKE    = False                         # True: 1 seed + tiny subset for a dry run


## 1 · Bootstrap — Drive, repo, deps


In [ ]:
import os, sys, subprocess
if MOUNT_DRIVE:
    try:
        from google.colab import drive; drive.mount('/content/drive')
        os.makedirs(DRIVE_DIR, exist_ok=True)
    except Exception as e:
        print('Drive mount skipped:', e); MOUNT_DRIVE = False
ROOT = '/content/HybridGuard'
if not os.path.isdir(ROOT):
    subprocess.run(['git','clone','--branch',BRANCH,REPO_URL,ROOT], check=True)
else:
    subprocess.run(['git','-C',ROOT,'fetch','origin'], check=True)
    subprocess.run(['git','-C',ROOT,'checkout',BRANCH], check=True)
    subprocess.run(['git','-C',ROOT,'pull','origin',BRANCH], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',ROOT+'[full]'], check=True)
os.chdir(ROOT); sys.path.insert(0, ROOT+'/src')
print('repo at', ROOT, '| branch', BRANCH)


## 2 · Imports + determinism


In [ ]:
import numpy as np, pandas as pd
from datetime import date
from hybridguard.canopi import data as D, eval as EV
from hybridguard.canopi.train import load_config, set_determinism, train_one_seed
from hybridguard.canopi.encoders import get_encoder
from hybridguard.canopi.augment import TransformationBank, NLLBTranslator
from hybridguard.canopi.attacks import SemanticRewriteAttacker
from hybridguard.canopi.runs import RunWriter
from hybridguard.canopi import metrics as M
cfg = load_config(CONFIG)
RUN_ID = RUN_ID or f"run_{cfg['name']}_{date.today().strftime('%Y%m%d')}"
RUN_ROOT = (DRIVE_DIR + '/runs') if MOUNT_DRIVE else 'runs'
writer = RunWriter(RUN_ID, root=RUN_ROOT)
print('RUN_ID', writer.run_id, '->', writer.dir)


## 3 · Data — reuse orchestrator protocol (60/20/20 @ 1337, leakage-checked)
If the orchestrator already populated `X_train/.../y_test` in globals, reuse
those; otherwise build them with `canopi.data` (identical seed/ratios).


In [ ]:
if all(v in globals() for v in ['X_train','y_train','X_val','y_val','X_test','y_test']):
    print('Reusing orchestrator splits from globals.')
    splits = None
else:
    df = D.load_xtram1()
    if SMOKE: df = df.sample(800, random_state=1337).reset_index(drop=True)
    splits, report = D.prepare_dataset(df, do_simhash=not SMOKE)
    assert report['clean'], f'LEAKAGE: {report}'
    writer.write_artifact_json('leakage_report', report)
    X_train,y_train = splits.xy('train'); X_val,y_val = splits.xy('val'); X_test,y_test = splits.xy('test')
    print('split', len(X_train), len(X_val), len(X_test), '| leakage clean:', report['clean'])
data = {'X_train':X_train,'y_train':y_train,'X_val':X_val,'y_val':y_val}


## 4 · Cross-corpus + multilingual eval sets (held out from training/τ)


In [ ]:
# deepset + JBB for threshold transfer (E4); NotInject for over-defense (E5).
extra = {}
for name, loader in [('deepset', D.load_deepset), ('jbb', D.load_jbb), ('notinject', D.load_notinject)]:
    try: extra[name] = loader(); print(name, len(extra[name]))
    except Exception as e: print('skip', name, e)
# Curated AR/ES (+NLLB MT). Reuse code/multilingual_injections.py if present.
ml = {}
try:
    sys.path.insert(0, ROOT+'/code'); from multilingual_injections import get_ml_sets
    raw = get_ml_sets(include_negatives=True)
    for lang,(txts,labs) in raw.items(): ml[lang] = (list(txts), list(np.asarray(labs).astype(int)))
    print('multilingual sets:', {k: len(v[0]) for k,v in ml.items()})
except Exception as e:
    print('curated AR/ES set unavailable (provide code/multilingual_injections.py):', e)
# MT diverse-8 panel: translate held-out English TEST positives into each language
# via NLLB (synthetic; AR/ES curated above stay the human-verified gold). Held out
# from training and from tau (tau frozen on English val).
try:
    from hybridguard.canopi.augment import NLLBTranslator
    LANGS = cfg.get('augment',{}).get('crosslingual', ['ar','es'])
    _tr = NLLBTranslator(cfg.get('augment',{}).get('nllb','facebook/nllb-200-distilled-600M'))
    en_pos = [t for t,y in zip(X_test,y_test) if int(y)==1][: (50 if SMOKE else 200)]
    mt = D.mt_multilingual_testset(_tr, en_pos, LANGS, max_n=(50 if SMOKE else 200))
    for k,(tx,lb) in mt.items(): ml[k] = (list(tx), list(lb))
    print('MT panel:', {k: len(v[0]) for k,v in mt.items()})
except Exception as e:
    print('MT multilingual panel skipped:', e)
if not ml:
    print('='*64); print('  E3 (multilingual HEADLINE / F9) WILL BE SKIPPED'); 
    print('  -> code/multilingual_injections.py not importable on this runtime.'); print('='*64)
else:
    print('E3 ready:', sum(len(v[0]) for v in ml.values()), 'multilingual prompts across', list(ml))


## 5 · Baselines (B1, B5; B2/B3 if available). B8 from a prior HG run if loaded.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC; from sklearn.linear_model import LogisticRegression
from hybridguard.canonicalize import canonicalize
def _canon(ts): return [canonicalize(t).canonical for t in ts]
baselines = {}
# B1 TF-IDF + LinearSVM (McNemar reference)
vec = TfidfVectorizer(max_features=5000, ngram_range=(1,2)).fit(_canon(X_train))
svm = LinearSVC().fit(vec.transform(_canon(X_train)), y_train)
def b1_score(ts):
    d = svm.decision_function(vec.transform(_canon(ts))); return 1/(1+np.exp(-d))
baselines['B1_tfidf_svm'] = b1_score
# B5 plain embedding clf on E(c(x)) -- no-invariance floor
enc = get_encoder(cfg.get('encoder', {'backend':'auto'}))
lr = LogisticRegression(max_iter=1000).fit(enc.encode(X_train), y_train)
baselines['B5_embedding'] = lambda ts: lr.predict_proba(enc.encode(ts))[:,1]
# B2/B3 optional HF detectors
def _hf(model_id):
    # trust_remote_code=True: InjecGuard ships a custom model class; these are
    # explicit, published baselines so we consent non-interactively (no [y/N] prompt).
    from transformers import pipeline; pipe = pipeline('text-classification', model=model_id, truncation=True, max_length=256, trust_remote_code=True)
    pos = {'INJECTION','LABEL_1','jailbreak','unsafe'}
    def f(ts):
        out = pipe(_canon(list(ts)))
        return np.array([o['score'] if o['label'] in pos else 1-o['score'] for o in out])
    return f
for tag, mid in [('B2_deberta','protectai/deberta-v3-base-prompt-injection-v2'),('B3_injecguard','leolee99/InjecGuard')]:
    try: baselines[tag] = _hf(mid); print('loaded', tag)
    except Exception as e: print('skip', tag, e)
print('baselines:', list(baselines))


## 6 · Train CANOPI over 5 seeds (+ per-seed eval, frozen-τ)


In [ ]:
seeds = [SEEDS[0]] if SMOKE else SEEDS
for s in seeds:
    set_determinism(s)
    res = train_one_seed(cfg, data, s)
    tau = res.tau; model = res.model
    writer.write_artifact_json(f'accept_rates_seed{s}', res.accept_rates)
    if res.aligned_pairs:
        pd.DataFrame(res.aligned_pairs, columns=['en','foreign','lang']).to_csv(writer.dir/f'aligned_pairs_seed{s}.csv', index=False)
    p_val = model.score(X_val); p_test = model.score(X_test)
    # E1 main: CANOPI + baselines (same frozen-tau-on-val protocol)
    rows = [EV.main_metrics(y_val, p_val, y_test, p_test, 'CANOPI')]
    for name, fn in baselines.items():
        rows.append(EV.main_metrics(y_val, fn(X_val), y_test, fn(X_test), name))
    writer.write_seed(s, 'main_results', pd.DataFrame(rows))
    # E3 multilingual @ each detector's OWN frozen tau (HEADLINE / F9)
    if ml:
        ml_rows = [EV.crosslingual_at_tau({lg:(lab, model.score(tx)) for lg,(tx,lab) in ml.items()}, tau, 'CANOPI')]
        for bname, bfn in baselines.items():
            tb = M.threshold_at_fpr(y_val, bfn(X_val), 0.01)   # baseline's own 1%-FPR tau
            ml_rows.append(EV.crosslingual_at_tau({lg:(lab, bfn(tx)) for lg,(tx,lab) in ml.items()}, tb, bname))
        writer.write_seed(s, 'crosslingual', pd.concat(ml_rows, ignore_index=True))
    # --- detectors: CANOPI + in-run baselines, each at its OWN val-frozen 1%-FPR tau.
    # Running baselines through E4/E5/E7 tests whether the JOINT objective buys
    # operating-point robustness (threshold transfer / over-defense / adaptive) even
    # when raw cross-lingual recall ties B5.
    detectors = {'CANOPI': (model.score, tau)}
    for bname, bfn in baselines.items():
        detectors[bname] = (bfn, M.threshold_at_fpr(y_val, bfn(X_val), 0.01))
    # E4 threshold transfer (all detectors, each at its own tau)
    tt_rows = []
    for dname,(dfn,dtau) in detectors.items():
        corp = {c:(df_['label'].values, dfn(df_['text'].tolist())) for c,df_ in extra.items() if c in ('deepset','jbb')}
        if corp: tt_rows.append(EV.threshold_transfer(corp, dtau, dname))
    if tt_rows: writer.write_seed(s, 'threshold_transfer', pd.concat(tt_rows, ignore_index=True))
    # E5 over-defense on NotInject (all detectors)
    if 'notinject' in extra:
        nj = extra['notinject']
        od_rows = [EV.overdefense_sweep(y_val, dfn(X_val), nj['label'].values, dfn(nj['text'].tolist()), dname) for dname,(dfn,dtau) in detectors.items()]
        writer.write_seed(s, 'overdefense', pd.concat(od_rows, ignore_index=True))
    # E7 adaptive residual: rewrites target CANOPI's representation; all detectors
    # scored on the SAME rewrites (transfer setting) at their own tau.
    atk = SemanticRewriteAttacker(model.encoder, intent_floor=cfg.get('eval',{}).get('intent_floor',0.5))
    pos_test = [t for t,y in zip(X_test,y_test) if int(y)==1][:200]
    sweep = atk.strength_sweep(pos_test, [0,0.25,0.5,0.75,1.0])
    ad_rows = [{'model':dname,'rewrite_strength':float(stg),'recall_at_1pctfpr':float((dfn(rew)>=dtau).mean())}
               for dname,(dfn,dtau) in detectors.items() for stg,rew in sweep.items()]
    writer.write_seed(s, 'adaptive', pd.DataFrame(ad_rows))
    # E10 calibration
    writer.write_seed(s, 'calibration', pd.DataFrame([EV.calibration(y_test, p_test, 'CANOPI')]))
    # per-seed checkpoint to Drive (trained head only; frozen E not saved) so an
    # interruption never forces retraining a completed seed
    import torch; torch.save({'state_dict': model.net.state_dict(), 'tau': float(tau), 'in_dim': model.in_dim, 'config': cfg['name']}, str(writer.dir/f'seed_{s}'/'proj_head.pt'))
    print(f'seed {s}: tau={tau:.4f} val_R@1%={res.val_recall_at_1pct:.3f}  [checkpoint saved]')


## 7 · Aggregate across seeds + render LaTeX tables


In [ ]:
for name, keys in [('main_results',['model']), ('crosslingual',['model','lang']),
                   ('threshold_transfer',['model','corpus']), ('overdefense',['model','fpr_target']),
                   ('adaptive',['model','rewrite_strength']), ('calibration',['model'])]:
    agg = writer.aggregate(name, key_cols=keys)
    if agg is not None:
        writer.write_table(name, agg, caption=f'CANOPI {name} (mean±std, 5 seeds)', label=f'tab:canopi_{name}')
        print('aggregated', name, agg.shape)


## 8 · McNemar vs strongest neural baseline AND vs B5 (require p<0.01)


In [ ]:
if not SMOKE:
    set_determinism(SEEDS[0]); res = train_one_seed(cfg, data, SEEDS[0])
    tau = res.tau; pred_c = (res.model.score(X_test) >= tau).astype(int)
    mc = []
    for name, fn in baselines.items():
        tb = M.threshold_at_fpr(y_val, fn(X_val), 0.01); predb = (fn(X_test) >= tb).astype(int)
        r = M.mcnemar(y_test, pred_c, predb); mc.append({'vs':name, **r})
    writer.write_seed(SEEDS[0], 'mcnemar', pd.DataFrame(mc)); print(pd.DataFrame(mc))


## 9 · Export figure CSVs + run metadata


In [ ]:
import subprocess, json
meta = {'run_id': writer.run_id, 'config': cfg, 'seeds': seeds, 'protocol':'val-frozen tau @1%FPR',
        'branch': BRANCH}
try: meta['git_sha'] = subprocess.check_output(['git','-C',ROOT,'rev-parse','HEAD']).decode().strip()
except Exception: pass
writer.write_metadata(meta)
# Mirror key tables to paper/paper_v2_extract for the figure generators.
import shutil, os
for name in ['main_results','crosslingual','threshold_transfer','overdefense','adaptive','calibration']:
    src = writer.dir/'tables'/f'{name}.csv'
    if src.exists():
        dst = f'{ROOT}/paper/paper_v2_extract/canopi/{name}.csv'; os.makedirs(os.path.dirname(dst), exist_ok=True); shutil.copy(src, dst)
print('wrote', writer.dir/'run_metadata.json')


## 10 · Generate figures + print checklist


In [ ]:
subprocess.run([sys.executable, 'scripts/make_paper_figures.py'], cwd=ROOT)
import os
print('\n=== CANOPI run checklist ===')
for name in ['main_results','crosslingual','threshold_transfer','overdefense','adaptive','calibration']:
    ok = (writer.dir/'aggregated'/f'{name}_mean_std.csv').exists()
    print(f'  [{"x" if ok else " "}] {name}')
print('runs dir:', writer.dir)


## 11 · Export & download results (run ANYTIME — safe after an interruption)
Per-seed CSVs already stream to Drive as each seed finishes, so completed seeds
survive a crash. This cell additionally (a) copies the figures (written into the
ephemeral repo clone) into the Drive run dir, (b) zips the whole run, (c) saves the
zip to Drive, and (d) triggers a browser download. Re-run cells 0–2 then THIS cell
to grab partial results after a disconnect — RUN_ID is date-derived, so it points
at the same Drive run dir.


In [ ]:
import shutil, glob, os
# (a) persist figures + figure-CSV mirror into the Drive run dir (else lost on crash)
fig_dst = writer.dir / 'figures'; fig_dst.mkdir(parents=True, exist_ok=True)
for f in glob.glob(f'{ROOT}/paper/figures/canopi_*'): shutil.copy(f, fig_dst)
csv_src = f'{ROOT}/paper/paper_v2_extract/canopi'
if os.path.isdir(csv_src): shutil.copytree(csv_src, str(writer.dir/'figure_csvs'), dirs_exist_ok=True)
n_fig = len(list(fig_dst.glob('canopi_*')))
print(f'persisted {n_fig} figure files into {fig_dst}')
# (b,c) zip the whole run dir -> save the archive to Drive (survives runtime death)
out_root = DRIVE_DIR if MOUNT_DRIVE else '/content'
zip_path = shutil.make_archive(f'{out_root}/{writer.run_id}_export', 'zip',
                               root_dir=str(writer.dir.parent), base_dir=writer.run_id)
print(f'zipped -> {zip_path} ({os.path.getsize(zip_path)/1e6:.1f} MB)')
# (d) best-effort immediate browser download (interactive Colab only)
try:
    from google.colab import files; files.download(zip_path)
except Exception as e:
    print('Auto-download unavailable; the zip is safe on Drive at', zip_path, '|', e)
